In [3]:
import httpx
from bs4 import BeautifulSoup
import asyncio
import re
from urllib.parse import urlencode

In [4]:
data = [
    ('PRIMER_LEFT_INPUT', 'GGCAAATGGGTTTTCAAGATCC'),
    ('PRIMER_RIGHT_INPUT', 'GAATATTCATANNTCCANTACC'),
    ('ORGANISM', 'oomycetes (taxid:4762)'),
    ('ORGANISM', 'Rhodophyta (taxid:2763)'),
    ('PRIMER_SPECIFICITY_DATABASE', 'core_nt'),
    ('SEARCH_SPECIFIC_PRIMER', 'on')
]

response = httpx.post('https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi', content=urlencode(data), timeout=httpx.Timeout(30, read=30))

In [5]:
print(response.text)

<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html xmlns="http://www.w3.org/1999/xhtml">
<head>
<META HTTP-EQUIV=Refresh CONTENT="20; URL=https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi?ctg_time=1756888597&job_key=MznuFkciSoptsNq119X-h63O77WA3fSogQ">
<title>Primer-Blast results</title>
<!-- style -->
<meta http-equiv="Pragma" content="no-cache">
<link rel="stylesheet" type="text/css" href="css/uswds.min.css" media="screen" />
<!--<link rel="stylesheet"  type="text/css" href="https://www.ncbi.nlm.nih.gov/style-guide/static/nwds/css/nwds.css"/>-->
<link rel="stylesheet"  type="text/css" href="css/nwds.css"/>
<link rel="stylesheet" href="css/header.css"/>
<link rel="stylesheet" href="css/footer.css"/>
<link rel="stylesheet" href="https://use.fontawesome.com/releases/v5.5.0/css/all.css" crossorigin="anonymous"> <!-- Font Awesome icons -->
<link rel="stylesheet" type="text/css" href="css/main.css" me

In [6]:
soup = BeautifulSoup(response.text, "html.parser")

# Find an input with a specific name
time_input = soup.find("input", {"name": "ctg_time"})
job_key_input = soup.find("input", {"name": "job_key"})
time = time_input['value']
job_key = job_key_input["value"]

In [5]:
result_url = f'https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi?ctg_time={time}&job_key={job_key}'

In [7]:
def poll_for_results(time: str, job_key: str):
    return httpx.get('https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi', params={
        'ctg_time': time,
        'job_key': job_key
    })

In [ ]:
poll_response = poll_for_results(time, job_key)
while poll_response.status_code == 200 and poll_response.text.find('Detailed primer reports') == -1:
    poll_response = poll_for_results(time, job_key)
    await asyncio.sleep(15)

In [9]:
print(poll_response.text)

<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html xmlns="http://www.w3.org/1999/xhtml">
<head>
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta name="ncbitoggler" content="animation:'none'"/>
<meta name="referrer" content="origin-when-cross-origin" />
<meta name="ncbi_app" content="primerblast" />
<meta name="ncbi_pdid" content="blastresults" />
<meta name="ncbi_program" content="blastn" />
<meta name="ncbi_db" content="core_nt" />
<meta name="ncbi_sessionid" content="0751197C8B7FE151_0000SID" />
<meta name="ncbi_phid" content="31C9CC258B7FE1510000000000000001" />
<script type="text/javascript">    var ncbi_startTime = new Date(); </script>
<title>Primer-Blast results</title>
<!-- style -->
<meta http-equiv="Pragma" content="no-cache">
<link rel="stylesheet" type="text/css" href="css/uswds.min.css" media="screen" />
<!--<link rel="stylesheet"  type="text/css" href="https://www.ncbi.

In [10]:
def parse_line(line):
    parts = line.split()
    if len(parts) < 4:
        return None
    label = " ".join(parts[:-3])  # handle labels with spaces
    return {
        "label": label,
        "start": int(parts[-3]),
        "sequence": parts[-2],
        "end": int(parts[-1]),
    }

In [11]:
from collections import defaultdict


bs = BeautifulSoup(poll_response.text, "html.parser")

# Find an input with a specific name
res_accessions = bs.find_all("a", {"target": "new_entrez"})

rows = []

for accession in res_accessions:
    hit_pre = accession.find_next_sibling('pre').text
    product_length = re.search(r"product length = (\d+)", hit_pre).group(1)
    lines = [parse_line(l) for l in hit_pre.splitlines() if l.strip() and "product length" not in l]
    parsed = [l for l in lines if l]

    # Handle duplicates
    label_counter = defaultdict(int)
    row = {"product_length": product_length}

    for entry in parsed:
        base_label = entry["label"].replace(" ", "_").lower()
        label_counter[base_label] += 1
        label = f"{base_label}_{label_counter[base_label]}"  # make unique
            
        row[f"{label}_start"] = entry["start"]
        row[f"{label}_seq"] = entry["sequence"]
        row[f"{label}_end"] = entry["end"]
    
    # accession
    row['accession'] = str(accession.text)
    acc_link = accession.attrs['href']
    match = re.search(r"id=(\d+)", acc_link)
    row['uid'] = match.group(1)

    rows.append(row)
rows    

[{'product_length': '334',
  'forward_primer_1_start': 1,
  'forward_primer_1_seq': 'GGCAAATGGGTTTTCAAGATCC',
  'forward_primer_1_end': 22,
  'template_1_start': 2,
  'template_1_seq': '......................',
  'template_1_end': 23,
  'reverse_primer_1_start': 1,
  'reverse_primer_1_seq': 'GAATATTCATANNTCCANTACC',
  'reverse_primer_1_end': 22,
  'template_2_start': 335,
  'template_2_seq': '...........AC....A....',
  'template_2_end': 314,
  'accession': 'MN164538.1',
  'uid': '1790965014'},
 {'product_length': '334',
  'forward_primer_1_start': 1,
  'forward_primer_1_seq': 'GGCAAATGGGTTTTCAAGATCC',
  'forward_primer_1_end': 22,
  'template_1_start': 2,
  'template_1_seq': '......................',
  'template_1_end': 23,
  'reverse_primer_1_start': 1,
  'reverse_primer_1_seq': 'GAATATTCATANNTCCANTACC',
  'reverse_primer_1_end': 22,
  'template_2_start': 335,
  'template_2_seq': '...........AC....A....',
  'template_2_end': 314,
  'accession': 'OR250019.1',
  'uid': '2771399368'},
 {

In [12]:
import csv
if len(rows) > 0:
    with open("primer-blast.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        for row in rows:
            writer.writerow(row)

In [ ]:
from src import primer

organisms = [
    4762,
    2763,
]
primer.run_primer_blast('GGCAAATGGGTTTTCAAGATCC', 'GAATATTCATANNTCCANTACC', organisms, "primer-blast.csv")

In [ ]:
from httpx import AsyncClient

async def primer_blast_siblings(tax_id: int, api_client: AsyncClient,):
    parent_response = await api_client.post('https://api.ncbi.nlm.nih.gov/datasets/v2/taxonomy/dataset_report', json={
        "taxons": [
            str(tax_id)
        ]
    })
    parent_data = parent_response.json()['reports']
    if len(parent_data) > 0:
        sisters = parent_data[0]['taxonomy']['children']
        assert tax_id in sisters
        sisters.remove(tax_id)
        print(sisters)
        primer.run_primer_blast('GGCAAATGGGTTTTCAAGATCC', 'GAATATTCATANNTCCANTACC', sisters, f"primer-blast-{tax_id}.csv")


In [2]:
from src import taxonomy
with open('data/cox2_th_cutadapt/host-tax-0.xml', 'r') as f:
    taxa = taxonomy.parse_xml(f.read())
    print(taxa)

[TaxonData(tax_id=4081, name='Solanum lycopersicum', rank='species', parent_tax_id=49274, lineage=[Taxon(tax_id=131567, name='cellular organisms', rank='cellular root'), Taxon(tax_id=2759, name='Eukaryota', rank='domain'), Taxon(tax_id=33090, name='Viridiplantae', rank='kingdom'), Taxon(tax_id=35493, name='Streptophyta', rank='phylum'), Taxon(tax_id=131221, name='Streptophytina', rank='subphylum'), Taxon(tax_id=3193, name='Embryophyta', rank='clade'), Taxon(tax_id=58023, name='Tracheophyta', rank='clade'), Taxon(tax_id=78536, name='Euphyllophyta', rank='clade'), Taxon(tax_id=58024, name='Spermatophyta', rank='clade'), Taxon(tax_id=3398, name='Magnoliopsida', rank='class'), Taxon(tax_id=1437183, name='Mesangiospermae', rank='clade'), Taxon(tax_id=71240, name='eudicotyledons', rank='clade'), Taxon(tax_id=91827, name='Gunneridae', rank='clade'), Taxon(tax_id=1437201, name='Pentapetalae', rank='clade'), Taxon(tax_id=71274, name='asterids', rank='clade'), Taxon(tax_id=91888, name='lamiids',

In [ ]:
from src import taxonomy, request, primer
from httpx import AsyncClient
from dotenv import load_dotenv
import os

load_dotenv()

# Get lineage information of target clade
client = request.HttpClient()
tax_id = await taxonomy.get_taxid_by_name('Oomycota', client)
if tax_id:
    #xml = await client.efetch_by_id('taxonomy', str(tax_id))
    api_client = AsyncClient(headers={'api-key': os.environ.get('NCBI_API_KEY')})
    response = await api_client.post('https://api.ncbi.nlm.nih.gov/datasets/v2/taxonomy/dataset_report', json={
        "taxons": [
            str(tax_id)
        ]
    })
    data = response.json()['reports']
    if len(data) > 0:
        parents = data[0]['taxonomy']['parents']
        if len(parents) > 0:
            # Remove root node
            parents.remove(1)
            parent = parents[-1]
            # get sister clades
            parent_response = await api_client.post('https://api.ncbi.nlm.nih.gov/datasets/v2/taxonomy/dataset_report', json={
                "taxons": [
                    str(parent)
                ]
            })
            parent_data = parent_response.json()['reports']
            if len(parent_data) > 0:
                sisters = parent_data[0]['taxonomy']['children']
                assert tax_id in sisters
                print(sisters)
                sisters.remove(tax_id)
                print(sisters)
                primer.run_primer_blast('GGCAAATGGGTTTTCAAGATCC', 'GAATATTCATANNTCCANTACC', sisters, "primer-blast-api-test.csv")


[77695, 46625, 2683627, 2587759, 4762, 2696291, 3111412, 42382, 2696290, 1892577, 54900, 2686031, 2683628, 238765]
[77695, 46625, 2683627, 2587759, 2696291, 3111412, 42382, 2696290, 1892577, 54900, 2686031, 2683628, 238765]
https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi?ctg_time=1757578198&job_key=5O47UwM_DpcpqRSsGcwwnmPXIaxOxDqxTw


In [ ]:
from src import taxonomy, request, primer

client = request.HttpClient()
tax_id = await taxonomy.get_taxid_by_name('Oomycota', client)
tax_xml = await client.efetch_by_id('taxonomy', str(tax_id))
tax_info = taxonomy.parse_xml(tax_xml)[0]
print(tax_info)

# lineage is ordered from top to bottom, but we want to start from the bottom
lineage = list(reversed(tax_info.lineage))
taxon = taxonomy.Taxon(TaxId=tax_info.tax_id, ScientificName=tax_info.name, Rank=tax_info.rank)
for parent, exclude_child in zip(lineage, [taxon, *lineage[:-1]]):
    sisters = await taxonomy.get_sisters(parent.tax_id, exclude_child.tax_id)
    print(sisters)
    for sister in sisters:
        primer.run_primer_blast('GGCAAATGGGTTTTCAAGATCC', 'GAATATTCATANNTCCANTACC', [sister], f"primer-blast-{sister}.csv")

tax_id=4762 name='Oomycota' rank='phylum' parent_tax_id=33634 lineage=[Taxon(tax_id=131567, name='cellular organisms', rank='cellular root'), Taxon(tax_id=2759, name='Eukaryota', rank='domain'), Taxon(tax_id=2698737, name='Sar', rank='clade'), Taxon(tax_id=33634, name='Stramenopiles', rank='clade')]
[77695, 46625, 2683627, 2587759, 2696291, 3111412, 42382, 2696290, 1892577, 54900, 2686031, 2683628, 238765]
https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi?ctg_time=1757582213&job_key=7-UyyLkAtKiTkiSXKfcApVPsEZd-_wqKfw
https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi?ctg_time=1757582255&job_key=q6F0HEQ3SZ9uoVOkXsR3liTfZqQJzH25CA
https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi?ctg_time=1757582296&job_key=GhDHPU0BQKlnk9CW3fb0pKft5ZaK_v6Liw
https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi?ctg_time=1757582337&job_key=h41YMGg3ZZ9CoX-kcsRblgjfSqQlzFG5JA
https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi?ctg_time=1757582378&job

AttributeError: 'NoneType' object has no attribute 'text'

In [1]:
from src import primer
primer.run_primer_blast('GGCAAATGGGTTTTCAAGATCC', 'GAATATTCATANNTCCANTACC', [554915], f"primer-blast-{554915}.csv")

The primer specificity cannot be determined as sequences for the selected organism (Amoebozoa) are not present in selected database: primer-blast-554915.csv.
 


In [ ]:
from src import primer

await primer.find_off_targets('GGCAAATGGGTTTTCAAGATCC', 'GAATATTCATANNTCCANTACC', 'Oomycota', 'data/cox2_th_cutadapt/primer-blast-outgroups.csv')

https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi?ctg_time=1757602215&job_key=zMYR6u3m4E7HcOV16BXBR5IO0HW_Hctovg
https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi?ctg_time=1757602256&job_key=w8ke5eLm707IcOp15xXOR50O33WwHcRosQ
https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi?ctg_time=1757602297&job_key=kZtOJlQdWbV-i0OOTu5nvDT1do4Z5m2TGA
https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi?ctg_time=1757602338&job_key=2tAH_etn5s_B9Xbwe5BSwgGLQ_AsmFjtLQ
https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi?ctg_time=1757602379&job_key=ioBXrKvmpk6BcKN1rhWHR9QOlnX5HY1o-A
https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi?ctg_time=1757602420&job_key=kJpNtrHmvE6bcLl1tBWdR84OjHXjHZdo4g
https://www.ncbi.nlm.nih.gov/tools/primer-blast/primertool.cgi?ctg_time=1757602462&job_key=fHahWl3mUE53cFV1WBVxRyIOYHUPHXtoDg
The primer specificity cannot be determined as sequences for the selected organism (Gyrista) are not present in select